In [8]:
import os
import re
import json
import math

from IPython.display import JSON

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from google import genai
from google.genai import types
from ydata_profiling import ProfileReport

# Loading our python file

from scripts import cleaner
from scripts import runner

from scripts import report_compressor as rc

In [9]:
# client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

### Step 1: Loading the dataset and Metadata

In [10]:

# df = pd.read_csv(r"C:\Users\shubh\Desktop\Auto EDA\notebooks\Zomato Restaurant Dataset.csv")

df = pd.read_csv(r"C:\Users\shubh\Desktop\Auto EDA\datasets\harry_potter_school_log_messy.csv")

def get_dataframe_metadata(df):

    metadata = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "columns": {}
    }

    for column in df.columns:

        metadata["columns"][column] = {
            "dtype": str(df[column].dtype),
            "missing_count": int(df[column].isna().sum()),
            "unique_count": int(df[column].nunique()),
            "sample_values": (
                df[column]
                .dropna()
                .head(5)
                .tolist()
            )
        }

    return metadata

metadata = get_dataframe_metadata(df)

### Step 2: Generates Cleaning Step from Y-Profiling Report (somewhere else)

In [11]:

# calls cleaner.py 

json_report = r"C:\Users\shubh\Desktop\Auto EDA\JSON reports\compressed_report.json"

cleaning_steps = cleaner.get_cleaning_recommendations(json_report)

print(cleaning_steps)

Based on the provided ydata-profiling report, here is a summary of the data quality issues and recommended actions:

*   **Missing Values**
    *   *Columns:* `Student_Name`, `House`, `Subject`, `Class_Date`, `Score`, `Attendance_Status`, `Teacher`
    *   *Issue:* Multiple columns contain missing values (e.g., 6 missing values in `Subject`, 4 in `Class_Date`, 3 in `Score`, 2 in `Student_Name`, `House`, and `Teacher`, and 1 in `Attendance_Status`).
    *   *Recommended Action:* Impute missing values where appropriate or drop the affected rows/columns depending on analysis requirements.

*   **Duplicate Rows**
    *   *Columns:* Entire dataset
    *   *Issue:* The dataset contains 6 duplicate rows (1.19%).
    *   *Recommended Action:* Remove the duplicate rows.

*   **Incorrect Data Types / Formatting**
    *   *Columns:* `Score`, `Class_Date`, `Year_Level`
    *   *Issue:* 
        *   `Score` is read as text, contains formatting anomalies (e.g., trailing `.0.0`), and invalid values (

### Step 3: Takes Cleaning Steps and Generates Python Code for Them

In [12]:

# stores generated python code for cleaning functions in a variable cleaning_functions_json

cleaning_functions_json = cleaner.generate_cleaning_functions(cleaning_steps, metadata)

print(cleaning_functions_json)

{
    "functions": [
        {
            "name": "remove_duplicate_rows",
            "columns": [],
            "action": "drop_duplicates",
            "reason": "The dataset contains 6 duplicate rows which should be removed.",
            "requires_review": false,
            "code": "def remove_duplicate_rows(df):\n    return df.drop_duplicates()"
        },
        {
            "name": "clean_student_name_whitespace",
            "columns": ["Student_Name"],
            "action": "strip_whitespace",
            "reason": "Student_Name contains leading whitespace.",
            "requires_review": false,
            "code": "def clean_student_name_whitespace(df):\n    df = df.copy()\n    if 'Student_Name' in df.columns:\n        df['Student_Name'] = df['Student_Name'].str.strip()\n    return df"
        },
        {
            "name": "clean_score_formatting",
            "columns": ["Score"],
            "action": "clean_and_cast_numeric",
            "reason": "Score is read a

### Step 4: Runs the Generates Function using Runner.py


In [13]:

# runs cleaning functions and returns cleaned dataframe

cleaned_df = runner.execute_steps(cleaning_functions_json, df, metadata) 

[SUCCESS] remove_duplicate_rows ran successfully.
[SUCCESS] clean_student_name_whitespace ran successfully.
[SUCCESS] clean_score_formatting ran successfully.
[SUCCESS] standardize_class_date ran successfully.
[SUCCESS] standardize_subject ran successfully.
[SUCCESS] standardize_attendance_status ran successfully.
[SUCCESS] standardize_teacher ran successfully.
Cleaning process completed.


In [ ]:
cleaned_df.head(5)

# cleaned_df.to_csv("cleaned_data.csv", index=False)

### Step 5: Testing code-repair function by passing error functions

In [17]:
error_function_json = r"""{
    "functions": [
        {
            "name": "remove_duplicate_rows",
            "columns": [],
            "action": "drop_duplicates",
            "reason": "The dataset contains 6 duplicate rows which should be removed.",
            "requires_review": false,
            "code": "def remove_duplicate_rows(df):\n    return df.drop_duplicate()"
        },
        {
            "name": "clean_student_name_whitespace",
            "columns": ["Student_Name"],
            "action": "strip_whitespace",
            "reason": "Student_Name contains leading whitespace.",
            "requires_review": false,
            "code": "def clean_student_name_whitespace(df):\n    df = df.copy()\n    if 'Student_Name' in df.columns:\n        df['Student_Name'] = df['StudentName'].str.strip()\n    return df"
        },
        {
            "name": "clean_score_formatting",
            "columns": ["Score"],
            "action": "clean_and_cast_numeric",
            "reason": "Score is read as text, contains formatting anomalies like trailing .0.0, and invalid out-of-range values.",
            "requires_review": true,
            "code": "import numpy as np\ndef clean_score_formatting(df):\n    df = df.copy()\n    if 'Score' in df.columns:\n        df['Score'] = df['Score'].astype(str).str.replace(r'\\.0+', '', regex=True)\n        df['Score'] = pd.to_numeric(df['Score'], errors='coerce')\n        df.loc[(df['Score'] < 0) | (df['Score'] > 100), 'Score'] = np.nan"
        },
        {
            "name": "standardize_class_date",
            "columns": ["Class_Date"],
            "action": "standardize_dates",
            "reason": "Class_Date uses mixed formats (dashes vs slashes) and contains text/invalid entries.",
            "requires_review": true,
            "code": "def standardize_class_date(df):\n    df = df.copy()\n    if 'Class_Date' in df.columns:\n    df['Class_Date'] = pd.to_datetime(df['Class_Date'].str.replace('/', '-', regex=False), errors='coerce')\n    return df"
        },
        {
            "name": "standardize_subject",
            "columns": ["Subject"],
            "action": "standardize_text",
            "reason": "Subject contains case inconsistencies, trailing spaces, and abbreviations.",
            "requires_review": true,
            "code": "def standardize_subject(df):\n    df = df.copy()\n    if 'Subject' in df.columns:\n        df['Subject'] = df['Subject'].str.strip().str.title()\n        df['Subject'] = df['Subject'].replace(subject_map)\n    return df"
        },
        {
            "name": "standardize_attendance_status",
            "columns": ["Attendance_Status"],
            "action": "standardize_categorical",
            "reason": "Attendance_Status contains inconsistent casing and alternative values like 'here'.",
            "requires_review": true,
            "code": "def standardize_attendance_status(df, mode):\n    df = df.copy()\n    if 'Attendance_Status' in df.columns:\n        df['Attendance_Status'] = df['Attendance_Status'].str.strip().str.title()\n        df['Attendance_Status'] = df['Attendance_Status'].replace({'Here': 'Present'})\n    return df"
        },
        {
            "name": "standardize_teacher",
            "columns": ["Teacher"],
            "action": "standardize_text",
            "reason": "Teacher has mixed casing across records.",
            "requires_review": false,
            "code": "def standardise_teacher(df):\n    df = df.copy()\n    if 'Teacher' in df.columns:\n        df['Teacher'] = df['Teacher'].str.strip().str.title()\n    return df"
        }
    ]
}"""

runner.execute_steps(error_function_json, df, metadata)

[FAILED] remove_duplicate_rows — AttributeError: 'DataFrame' object has no attribute 'drop_duplicate'
Calling Code Repair function to fix the error...
[SUCCESS] remove_duplicate_rows ran successfully after repair.
[FAILED] clean_student_name_whitespace — KeyError: 'StudentName'
Calling Code Repair function to fix the error...
[SUCCESS] clean_student_name_whitespace ran successfully after repair.
[FAILED] clean_score_formatting — TypeError: clean_score_formatting returned NoneType, expected DataFrame
Calling Code Repair function to fix the error...
[SUCCESS] clean_score_formatting ran successfully after repair.
[FAILED] standardize_class_date — IndentationError: expected an indented block after 'if' statement on line 3 (<string>, line 4)
Calling Code Repair function to fix the error...
[SUCCESS] standardize_class_date ran successfully after repair.
[FAILED] standardize_subject — NameError: name 'subject_map' is not defined
Calling Code Repair function to fix the error...
[SUCCESS] stand

,Student_ID,Student_Name,House,Year_Level,Subject,Class_Date,Score,Attendance_Status,Teacher
0,STU0001,Viktor Malfoy,Gryffindor,6,Transfiguration,1991-10-28,52.0,Late,Flitwick
1,STU0002,Draco Brown,Gryffindor,5,Herbology,1991-09-08,46.0,Present,Snape
2,STU0003,Fred Jordan,Gryffindor,5,Charms,1992-02-14,88.0,Present,Sinistra
3,STU0004,Padma Finnigan,Gryffindor,7,Charms,1991-12-18,78.0,Absent,Sinistra
4,STU0005,Luna Chang,Ravenclaw,1,Potions,NaT,80.0,Absent,Sprout
...,...,...,...,...,...,...,...,...,...
495,STU0496,Oliver Malfoy,Ravenclaw,1,Potions,1991-10-21,60.0,Absent,Sprout
496,STU0497,Oliver Chang,Ravenclaw,6,Potions,1991-10-12,40.0,Late,Mcgonagall
497,STU0498,Pansy Chang,Hufflepuff,3,Charms,1991-09-04,46.0,Present,Snape
498,STU0499,Lavender Johnson,Slytherin,1,Defence Against The Dark Arts,1991-09-26,67.0,Late,Snape
